# 7.2 · 分类评估指标 / Classification Metrics

> **课程定位 / Where this fits**
> 第 2 课，**Part 7 · 模型评估与优化**。
> Lesson 2, **Part 7 · Model Evaluation & Tuning**.
>
> 5.1 已经建立了分类指标的基础（混淆矩阵、P/R/F1、ROC/PR）。这一课**系统化并补全**：加上 **MCC、对数损失(log loss)、多分类的 micro/macro/weighted 平均**，并把"何时看哪个指标"讲成一张决策图。分类指标是数据岗面试的**绝对高频区**。
> 5.1 laid the foundation (confusion matrix, P/R/F1, ROC/PR). This lesson **systematizes and completes** it: adding **MCC, log loss, and micro/macro/weighted averaging for multi-class**, and turning "which metric when" into a decision chart. Classification metrics are a **top interview area**.
>
> 💼 **实战/面试视角**："accuracy/precision/recall/F1/AUC 区别和选择 / 不平衡看什么 / 多分类怎么平均" 几乎必问。
> 💼 **Practical/interview angle:** "differences and choice among accuracy/precision/recall/F1/AUC / what to use under imbalance / multi-class averaging" — near-guaranteed.

> 📐 **符号约定 / Notation**
> - TP/FP/FN/TN —— 真阳/假阳/假阴/真阴 / confusion-matrix cells
> - $p_i$ —— 预测为正类的概率 / predicted positive probability

> 💡 **面试相关 / Interview-relevant**
> - "precision/recall/F1 的定义与权衡"（出镜率 ★★★★★）
> - "ROC-AUC vs PR-AUC, 不平衡用哪个"（★★★★★）
> - "MCC 是什么 / 为什么比 F1/accuracy 更平衡"（★★★★）
> - "对数损失(log loss)衡量什么"（★★★★）
> - "多分类 micro vs macro 平均"（★★★★★）

---

## 学习目标 / Learning Objectives

1. 从混淆矩阵复盘 P/R/F1/特异度（接 5.1）。
   Recap P/R/F1/specificity from the confusion matrix (continuing 5.1).
2. 理解 **ROC-AUC vs PR-AUC** 的选择。
   Understand the ROC-AUC vs PR-AUC choice.
3. 掌握 **MCC** 与**对数损失**这两个被低估的指标。
   Master two underused metrics: MCC and log loss.
4. 掌握多分类的 **micro/macro/weighted** 平均。
   Master micro/macro/weighted averaging for multi-class.
5. 用一张决策图**按场景选指标**。
   Pick the metric per scenario via a decision chart.

## 目录 / TOC
1. [先建直觉 + 混淆矩阵复盘 ⭐](#1)
2. [🩺 数据 + P/R/F1 全家桶 ⭐](#2)
3. [ROC-AUC vs PR-AUC ⭐](#3)
4. [MCC + 对数损失 ⭐](#4)
5. [多分类平均：micro/macro ⭐](#5)
6. [选指标决策图 + 小结](#6)


<a id="1"></a>
## 1. 先建直觉 + 混淆矩阵复盘 ⭐ / Intuition & Confusion-Matrix Recap

一切分类指标都从**混淆矩阵**的四个数派生（5.1 详讲）：TP（真阳）、FP（假阳，误报）、FN（假阴，漏报）、TN（真阴）。关键洞察：**准确率会骗人**（不平衡时全猜多数类就很高），所以我们需要一整套指标，从不同角度刻画"模型到底好不好"。
Every classification metric derives from the four cells of the **confusion matrix** (detailed in 5.1): TP, FP (false alarm), FN (miss), TN. The key insight: **accuracy lies** (under imbalance, predicting the majority scores high), so we need a whole suite of metrics describing "how good is it" from different angles.

回顾核心定义：
Recall the core definitions:
- **Precision** $=\frac{TP}{TP+FP}$：报警准不准（误报代价高时关心）。
  Precision: how trustworthy the alarms are (matters when false alarms are costly).
- **Recall（召回/敏感度）** $=\frac{TP}{TP+FN}$：抓得全不全（漏报代价高时关心）。
  Recall (sensitivity): how many positives we catch (matters when misses are costly).
- **F1** $=2\frac{PR}{P+R}$：P 与 R 的调和均值（要平衡两者时用）。
  F1: harmonic mean of P and R (when balancing both).
- **特异度 Specificity** $=\frac{TN}{TN+FP}$：负类抓得全不全。
  Specificity: how well negatives are identified.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import confusion_matrix
sns.set_theme(style="whitegrid")

data = load_breast_cancer()
X_tr, X_te, y_tr, y_te = train_test_split(data.data, data.target, test_size=0.3,
                                          stratify=data.target, random_state=0)
clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)).fit(X_tr, y_tr)
y_pred = clf.predict(X_te)
y_proba = clf.predict_proba(X_te)[:, 1]

tn, fp, fn, tp = confusion_matrix(y_te, y_pred).ravel()    # 展平成四个数
print(f"混淆矩阵 confusion matrix: TP={tp}, FP={fp}, FN={fn}, TN={tn}")
print(f"precision = TP/(TP+FP) = {tp/(tp+fp):.3f}")
print(f"recall    = TP/(TP+FN) = {tp/(tp+fn):.3f}")
print(f"specificity= TN/(TN+FP)= {tn/(tn+fp):.3f}")


<a id="2"></a>
## 2. 数据 + P/R/F1 全家桶 ⭐ / The Full P/R/F1 Family

`classification_report` 一次给出每个类的 precision/recall/F1 和支持数。**注意"正类是谁"很重要**——在癌症数据里 1=良性，但我们真正关心的是不要漏掉恶性，所以实务中应把"恶性"设为正类并优化 recall（接 5.1 的业务讨论）。
`classification_report` gives precision/recall/F1 and support per class at once. **Note "which class is positive" matters** — here 1=benign, but we really care about not missing malignant, so in practice set "malignant" as positive and optimize recall (the business point from 5.1).


In [ ]:
from sklearn.metrics import classification_report, accuracy_score, balanced_accuracy_score

print(classification_report(y_te, y_pred, target_names=["恶性 malignant","良性 benign"], digits=3))
print(f"accuracy          = {accuracy_score(y_te, y_pred):.3f}")
# balanced accuracy = 各类 recall 的平均, 不平衡时比 accuracy 公平 / mean of per-class recall
print(f"balanced accuracy = {balanced_accuracy_score(y_te, y_pred):.3f}  (各类 recall 的平均, 抗不平衡)")
print("\n💡 报告里 macro avg = 各类指标简单平均(每类等权); weighted avg = 按样本数加权")


<a id="3"></a>
## 3. ROC-AUC vs PR-AUC ⭐ / ROC-AUC vs PR-AUC

P/R/F1 都依赖**一个固定阈值(0.5)**。**ROC-AUC** 和 **PR-AUC** 则**扫遍所有阈值**，衡量模型的**排序能力**（与阈值无关）。核心区别（面试高频）：
P/R/F1 depend on **one fixed threshold (0.5)**. **ROC-AUC** and **PR-AUC** sweep **all thresholds**, measuring the model's **ranking ability** (threshold-free). The key difference (frequently asked):
- **ROC-AUC**：TPR vs FPR 曲线下面积。FPR 的分母含巨大的 TN，所以**极端不平衡时 ROC 会过度乐观**（大量假阳也只让 FPR 微增）。
  **ROC-AUC:** area under TPR-vs-FPR. FPR's denominator includes huge TN, so **ROC is over-optimistic under extreme imbalance** (many false positives barely move FPR).
- **PR-AUC**：precision vs recall 曲线下面积，**完全不看 TN**，只盯正类——**不平衡问题的首选**。
  **PR-AUC:** area under precision-vs-recall, **ignores TN entirely**, focusing on the positive class — **preferred under imbalance**.


In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, precision_recall_curve

# 造一个极端不平衡场景对比 ROC vs PR / extreme imbalance to contrast
rng = np.random.default_rng(0)
n_pos, n_neg = 30, 3000
s = np.r_[rng.normal(1.0, 1, n_pos), rng.normal(0.0, 1, n_neg)]   # 正类分数略高
yt = np.r_[np.ones(n_pos), np.zeros(n_neg)]

fig, axes = plt.subplots(1, 2, figsize=(12, 4.3))
fpr, tpr, _ = roc_curve(yt, s)
axes[0].plot(fpr, tpr); axes[0].plot([0,1],[0,1],"k--")
axes[0].set_title(f"ROC (AUC={roc_auc_score(yt, s):.2f}) — 看着不错(乐观)")
axes[0].set_xlabel("FPR"); axes[0].set_ylabel("TPR")
prec, rec, _ = precision_recall_curve(yt, s)
axes[1].plot(rec, prec); axes[1].axhline(n_pos/(n_pos+n_neg), color="k", ls="--", label=f"基线={n_pos/(n_pos+n_neg):.1%}")
axes[1].set_title(f"PR (AP={average_precision_score(yt, s):.2f}) — 揭示真实困难")
axes[1].set_xlabel("recall"); axes[1].set_ylabel("precision"); axes[1].legend()
plt.tight_layout(); plt.show()
print(f"1% 正类: ROC-AUC={roc_auc_score(yt, s):.2f}(乐观) 但 PR-AUC={average_precision_score(yt, s):.2f}(真实困难)")
print("→ 不平衡看 PR-AUC; ROC 被海量 TN 掩盖了 precision 问题(同 5.1 结论)")


<a id="4"></a>
## 4. MCC + 对数损失 ⭐ / MCC & Log Loss

两个被低估但很重要的指标：
Two underused but important metrics:
- **MCC（马修斯相关系数）**：用上混淆矩阵**全部四个数**的一个 $[-1,1]$ 相关系数。1=完美，0=随机，−1=完全反。它在**不平衡下比 F1/accuracy 更诚实**——因为 F1 忽略 TN、accuracy 被多数类主导，而 MCC 对四个格子平衡敏感。很多论文/竞赛用它做不平衡的单一指标。
  **MCC (Matthews correlation coefficient):** a $[-1,1]$ correlation using **all four cells**. 1=perfect, 0=random, −1=fully wrong. **More honest than F1/accuracy under imbalance** — F1 ignores TN and accuracy is majority-dominated, while MCC is balanced across all four cells. Often the single metric of choice for imbalanced tasks.
- **对数损失(log loss / 交叉熵)**：不看 0/1 预测，而看**预测概率有多准**。它**重罚自信的错误**（说 0.99 却错了，损失爆炸）。当你需要可信的概率（风控阈值、期望损失）时，用它衡量，并配合校准(7.9)。
  **Log loss (cross-entropy):** ignores the 0/1 prediction and measures **how accurate the predicted probabilities are**. It **heavily penalizes confident mistakes** (claiming 0.99 but wrong → exploding loss). Use it when you need trustworthy probabilities (risk thresholds, expected loss), with calibration (7.9).


In [ ]:
from sklearn.metrics import matthews_corrcoef, log_loss, f1_score

print(f"MCC      = {matthews_corrcoef(y_te, y_pred):.3f}  (用全部四格, [-1,1], 不平衡下更诚实)")
print(f"F1       = {f1_score(y_te, y_pred):.3f}  (忽略 TN)")
print(f"log loss = {log_loss(y_te, y_proba):.3f}  (越低越好, 看概率准不准)\n")

# 演示 log loss 重罚自信的错误 / log loss punishes confident wrong predictions
print("log loss 对'自信的错误'有多狠 (真实=1):")
for p in [0.9, 0.6, 0.4, 0.1, 0.01]:
    ll = log_loss([1], [p], labels=[0,1])
    tag = "对(自信)" if p>0.5 else "错"
    print(f"  预测概率 {p:.2f} ({tag}): log loss = {ll:.2f}")
print("→ 预测 0.01 却真值为1 → log loss 飙到 4.6; 越自信地错, 罚得越狠")


<a id="5"></a>
## 5. 多分类平均：micro/macro ⭐ / Multi-class Averaging

二分类的 P/R/F1 怎么推广到多分类？对每个类各算一份，再**汇总成一个数**——汇总方式决定了你在意什么（面试高频）：
How do binary P/R/F1 extend to multi-class? Compute one per class, then **aggregate into a single number** — the aggregation decides what you care about (frequently asked):
- **macro**：每类各算 F1 再**简单平均**（每类等权）→ **少数类和多数类一样重要**。关心稀有类时用。
  **macro:** per-class F1 then **simple average** (each class equal) → **minority counts as much as majority**. Use when rare classes matter.
- **micro**：把所有类的 TP/FP/FN **先汇总**再算（每**样本**等权）→ **被多数类主导**（多分类里 micro-F1 = accuracy）。
  **micro:** pool all classes' TP/FP/FN **first**, then compute (each **sample** equal) → **majority-dominated** (micro-F1 = accuracy in multi-class).
- **weighted**：按各类样本数加权的 macro。
  **weighted:** macro weighted by class support.


In [ ]:
from sklearn.datasets import load_wine
from sklearn.metrics import f1_score, precision_score

wine = load_wine()
Xw_tr, Xw_te, yw_tr, yw_te = train_test_split(wine.data, wine.target, test_size=0.3,
                                              stratify=wine.target, random_state=0)
clf_w = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)).fit(Xw_tr, yw_tr)
yw_pred = clf_w.predict(Xw_te)

print("Wine 三分类的 F1 不同平均方式:")
for avg in ["micro", "macro", "weighted"]:
    print(f"  F1 ({avg:<8}) = {f1_score(yw_te, yw_pred, average=avg):.3f}")
print(f"  accuracy        = {accuracy_score(yw_te, yw_pred):.3f}  (= micro-F1, 多分类下相等)")
print("\n本数据较均衡, 三者接近; 不平衡时 macro 会明显低于 micro(揭示少数类表现差)")


<a id="6"></a>
## 6. 选指标决策图 + 小结 / Decision Chart & Summary

把"何时看哪个"整理成决策图（最重要的实战产出）：
The decision chart for "which metric when" (the key practical takeaway):

| 场景 / Scenario | 主指标 / Primary |
|---|---|
| 类别均衡、关心整体对错 | accuracy / micro-F1 |
| 漏报代价高（癌症、欺诈漏抓）| **recall**（+ 调阈值 5.1）|
| 误报代价高（垃圾邮件误杀）| **precision** |
| P/R 都要平衡 | **F1** |
| 排序能力、均衡数据 | **ROC-AUC** |
| 排序能力、不平衡 | **PR-AUC** ⭐ |
| 不平衡下要单一可靠指标 | **MCC** |
| 需要可信概率（风控/期望损失）| **log loss** + 校准(7.9) |
| 多分类、在意稀有类 | **macro-F1** |

```
混淆矩阵四格 → precision(报准)/recall(抓全)/F1(调和)/specificity(负类)
阈值无关排序: ROC-AUC(均衡用) / PR-AUC(不平衡首选, 不看 TN)
MCC: 用全部四格的相关系数, 不平衡下比 F1/accuracy 诚实
log loss: 看概率准不准, 重罚自信的错误; 需可信概率时用+校准(7.9)
多分类: macro(类等权, 关注稀有类) / micro(样本等权=accuracy) / weighted
按业务成本选指标(漏报→recall, 误报→precision, 不平衡→PR-AUC/MCC)
```

### 💡 面试速查 / Interview cheat-sheet
1. **precision(报准) vs recall(抓全)**: 误报代价 vs 漏报代价的权衡。
   Precision vs recall: false-alarm cost vs missed-detection cost.
2. **ROC-AUC 均衡用, PR-AUC 不平衡用**(ROC 被海量 TN 蒙蔽)。
   ROC-AUC when balanced, PR-AUC when imbalanced.
3. **MCC** 用全部四格, 不平衡下比 F1/accuracy 更诚实。
   MCC uses all four cells, more honest than F1/accuracy under imbalance.
4. **log loss 看概率质量**, 重罚自信的错误; 配校准(7.9)。
   Log loss judges probability quality, punishing confident mistakes; pair with calibration.
5. **macro(类等权) vs micro(样本等权=accuracy)**; 关注稀有类用 macro。
   macro (class-equal) vs micro (sample-equal = accuracy); use macro for rare classes.

### 下一节 / Next
**7.3 偏差-方差权衡**——评估之后是诊断: 模型表现不好, 是欠拟合(高偏差)还是过拟合(高方差)? 学习曲线/验证曲线给出答案和对策。
**7.3 Bias-Variance** — after evaluation comes diagnosis: is poor performance underfitting (high bias) or overfitting (high variance)? Learning/validation curves give the answer and the fix.
